# WP1 — Sequence Dataset Build

Retrieve lipase homologs from UniProt, clean them (length / fragments / redundancy /
catalytic motif), annotate metadata, and write the curated FASTA + metadata table +
benchmark panel. See `enzymes.docx` §6.

In [1]:
# Make the src/ package importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from asr_poc.config import load_config
from asr_poc.io_utils import set_seed
cfg = load_config(ROOT / "config" / "target.yaml")
set_seed(cfg.run.seed)
cfg.paths.ensure_dirs()
print("Target family:", cfg.target.name)

Target family: lipase


## 1. Retrieve homologs from UniProt

In [2]:
from asr_poc import retrieval
raw = retrieval.retrieve(cfg)        # hits UniProt REST; respects retrieval.* config
raw.head()

09:12:24 [info     ] uniprot_fetch                  max_sequences=500 query=family:lipase AND reviewed:true stage=wp1.retrieval
09:12:25 [info     ] uniprot_fetched                n=500 stage=wp1.retrieval


,id,organism,length,fragment,ec_number,protein_name,annotation_score,sequence
0,A2VBC4,Polybia paulista (Neotropical social wasp) (Sw...,322,NaN,3.1.1.32,Phospholipase A1 (PLA1) (EC 3.1.1.32) (allerge...,5.0,MNFKYSILFICFGTLDRGLIPECPFNEYDILFFVYTRQQRDGIVLT...
1,A5PK46,Bos taurus (Bovine),469,NaN,3.1.1.26; 3.1.1.3,Pancreatic lipase-related protein 2 (PL-RP2) (...,5.0,MLPSWTIGLLLLATVRGKEICYEPFGCFSDEKPWTGILQRPLKLFP...
2,A8PUY1,Malassezia globosa (strain ATCC MYA-4612 / CBS...,304,NaN,3.1.1.-,Secreted mono- and diacylglycerol lipase LIP1 ...,5.0,MLFSRFVLLAFGSVAAVSASSIYARGRGGSSTDQPVANPYNTKEIS...
3,B0F2B4,Mus musculus (Mouse),945,NaN,NaN,Neuroligin 4-like (Neuroligin-4) (NL-4),5.0,MPAPVPALLCLALALASAQPSPPPPPPFPVVATNYGKLRGVRAALP...
4,D7EZN2,Sus scrofa (Pig),471,NaN,3.1.1.26; 3.1.1.3,Pancreatic lipase-related protein 2 (PL-RP2) (...,5.0,MLPSWTIGLLLLATVRGKEICYQPFGCFSDETPWARTCHWPFKLFP...


## 2. Clean + reduce redundancy + annotate

In [3]:
cleaned = retrieval.clean(raw, cfg)
cleaned = retrieval.reduce_redundancy(cleaned, cfg.curation.cluster_identity)
curated = retrieval.annotate(cleaned, cfg)
print(f"raw={len(raw)}  curated={len(curated)}")
curated[["id","organism","length","family","motif_start"]].head()

09:12:26 [info     ] cleaned                        after=330 before=500 dropped=170 stage=wp1.retrieval
09:12:29 [info     ] redundancy_reduced             after=315 before=330 identity=0.9 stage=wp1.retrieval
raw=500  curated=315


,id,organism,length,family,motif_start
0,A2VBC4,Polybia paulista (Neotropical social wasp) (Sw...,322,lipase,153
1,P54318,Rattus norvegicus (Rat),468,lipase,167
2,Q6P8U6,Mus musculus (Mouse),465,lipase,166
3,Q64573,Rattus norvegicus (Rat),561,lipase,218
4,Q64424,Myocastor coypus (Coypu) (Mus coypus),470,lipase,169


## 3. Write artifacts (curated FASTA, metadata, benchmark panel)

In [4]:
summary = retrieval.build_dataset(cfg, df_raw=raw)   # writes all WP1 outputs
summary

09:12:29 [info     ] cleaned                        after=330 before=500 dropped=170 stage=wp1.retrieval
09:12:32 [info     ] redundancy_reduced             after=315 before=330 identity=0.9 stage=wp1.retrieval
09:12:32 [info     ] wp1_complete                   benchmark=5 curated=315 raw=500 stage=wp1.retrieval


{'raw': 500, 'curated': 315, 'benchmark': 5}

In [5]:
import pandas as pd
pd.read_csv(cfg.paths.metadata_csv).head()

,id,organism,length,fragment,ec_number,protein_name,annotation_score,family,taxonomy,motif_start
0,A2VBC4,Polybia paulista (Neotropical social wasp) (Sw...,322,NaN,3.1.1.32,Phospholipase A1 (PLA1) (EC 3.1.1.32) (allerge...,5.0,lipase,Polybia paulista (Neotropical social wasp) (Sw...,153
1,P54318,Rattus norvegicus (Rat),468,NaN,3.1.1.26; 3.1.1.3,Pancreatic lipase-related protein 2 (PL-RP2) (...,5.0,lipase,Rattus norvegicus (Rat),167
2,Q6P8U6,Mus musculus (Mouse),465,NaN,3.1.1.3,Pancreatic triacylglycerol lipase (PL) (PTL) (...,5.0,lipase,Mus musculus (Mouse),166
3,Q64573,Rattus norvegicus (Rat),561,NaN,3.1.1.1,Liver carboxylesterase 1F (EC 3.1.1.1) (Carbox...,5.0,lipase,Rattus norvegicus (Rat),218
4,Q64424,Myocastor coypus (Coypu) (Mus coypus),470,NaN,3.1.1.26; 3.1.1.3,Pancreatic lipase-related protein 2 (PL-RP2) (...,5.0,lipase,Myocastor coypus (Coypu) (Mus coypus),169
